# Transform Races Data
 
1. Read bronze `races` table
2. Keep only the columns required for analytics (Drop `url` column)
3. Standardise column names using snake_case (`raceName` → `race_name`, `circuitId` → `circuit_id`)
4. Rename columns to make them more meaningful (`date` → `race_date`)
5. Remove duplicate records
6. Transform values of column `race_name` to Title Case
7. Write the transformed data to silver `races` table

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.races"
silver_table = f"{catalog_name}.{silver_schema}.races"

In [0]:
from pyspark.sql import functions as F

#### Step 1 - Read bronze `circuits` table

In [0]:
races_df = spark.table(bronze_table)

#### Step 2 - Keep only the columns required for analytics (Drop url column)

In [0]:
races_selected_df = races_df.select(
    F.col("season"),
    F.col("round"),
    F.col("raceName"),
    F.col("date"),
    F.col("circuitId"),
    F.col("injestion_timestamp"),
    F.col("source_file")
)

#### Step 3 & 4 - Standardise Column Names
- Standardise column names using snake_case (`circuitId` → `circuit_id`, `circuitName` → `circuit_name`)
- Rename columns to make them more meaningful (`lat` → `latitude`, `long` → `longitude`)

In [0]:
races_renamed_df = (
    races_selected_df
        .withColumnsRenamed({
            "circuitId":"circuit_id", 
            "raceName": "race_name", 
            "date": "race_date"
        })
)

#### Step 5 - Remove duplicate records

In [0]:
races_distinct_df = races_renamed_df.dropDuplicates(["season", "round"])
display(races_distinct_df)

#### Step 7 - Transform values of columns `race_name` to Title Case

In [0]:
races_final_df = (
    races_distinct_df
        .withColumn('race_name', F.initcap(F.col('race_name')))
)
display(races_final_df)

#### Step 8 - Write the transformed data to silver `circuits` table

In [0]:
(
    races_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))